In [0]:
%run "../common/define_path"

In [0]:
tracking_raw_path
tracking_br_path

In [0]:
def read_invoice_data():
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
    from pyspark.sql.functions import current_timestamp
  
    invoice_schema = StructType([
    StructField("invno",StringType()),
    StructField("invdate",StringType()),
    StructField("ord",StringType()),
    StructField("amount",StringType()),
    StructField("taxrate",StringType()),
    StructField("taxamount",StringType()),
    StructField("paymentdays",StringType()),
    StructField("nameofcustomer",StringType()),
    StructField("city",StringType()),
    StructField("state",StringType())
    ])

    rawinvoice_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option('cloudFiles.schemaLocation',f'{tracking_br_path}/schema')
        .option('header','true')
        .schema(invoice_schema)
        .load(tracking_raw_path+'/invoice/')
        .withColumn("ingestion_time",current_timestamp())
    )
    return rawinvoice_stream

In [0]:
def write_invoice_Data(StreamingDF):

    write_Stream = (StreamingDF.writeStream
.format('delta')
     .option("checkpointLocation",f'{tracking_br_path}/checkpoint/invoice/')
    .outputMode('append')
    .queryName('invoicewritestream')
    .trigger(availableNow=True)
    .toTable("dev.tracking_br.invoice"))
    
    write_Stream.awaitTermination()

In [0]:
df_invoice = read_invoice_data()
write_invoice_Data(df_invoice)